# Analytics Q&A Agent

A lightweight natural-language interface for product analytics. The system converts analytics questions into SQL, validates and executes them against a synthetic SQLite database, and asks for clarification when a question is ambiguous.

## 1. Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime, timedelta

np.random.seed(42)

print("Setup complete")

Setup complete


## Synthetic Dataset

This project uses synthetic product analytics data across three tables:

- `users`: user signup dates and countries
- `sessions`: user activity events
- `transactions`: purchase events

The dataset is intentionally small and controlled for fast local querying and reproducible evaluation.

## 2. Synthetic Dataset Generation

The synthetic dataset contains 5,000 users, approximately 40,000 sessions, and 8,000 transactions. A fixed random seed makes the dataset reproducible.

In [2]:
# -------------------------
# 1. Generate users
# -------------------------

n_users = 5000

start_date = pd.Timestamp("2026-01-01")
end_date = pd.Timestamp("2026-05-31")

signup_dates = start_date + pd.to_timedelta(
    np.random.randint(0, (end_date - start_date).days + 1, n_users),
    unit="D"
)

users = pd.DataFrame({
    "user_id": range(1, n_users + 1),
    "signup_date": signup_dates,
    "country": np.random.choice(
        ["US", "India", "UK", "Canada", "Germany"],
        size=n_users,
        p=[0.30, 0.30, 0.15, 0.15, 0.10]
    )
})

# -------------------------
# 2. Generate sessions
# -------------------------

session_rows = []
session_id = 1
data_end_date = pd.Timestamp("2026-06-30")

for _, user in users.iterrows():
    active_days = np.random.randint(1, 16)

    max_days = (data_end_date - user["signup_date"]).days

    chosen_days = np.random.choice(
        range(max_days + 1),
        size=min(active_days, max_days + 1),
        replace=False
    )

    for day in chosen_days:
        session_rows.append({
            "session_id": session_id,
            "user_id": user["user_id"],
            "session_date": user["signup_date"] + pd.Timedelta(days=int(day))
        })
        session_id += 1

sessions = pd.DataFrame(session_rows)

# -------------------------
# 3. Generate transactions
# -------------------------

n_transactions = 8000

transactions = pd.DataFrame({
    "transaction_id": range(1, n_transactions + 1),
    "user_id": np.random.choice(users["user_id"], n_transactions),
    "amount": np.round(np.random.uniform(5, 200, n_transactions), 2),
    "transaction_date": pd.to_datetime(
        np.random.choice(pd.date_range("2026-01-01", "2026-06-30"), n_transactions)
    )
})

print("Users:", len(users))
print("Sessions:", len(sessions))
print("Transactions:", len(transactions))

Users: 5000
Sessions: 40088
Transactions: 8000


In [3]:
print("US users:", (users["country"] == "US").sum())
print("Date range:", sessions["session_date"].min(), "to", sessions["session_date"].max())
print("Unique active users:", sessions["user_id"].nunique())

display(users.head())
display(sessions.head())
display(transactions.head())

US users: 1515
Date range: 2026-01-03 00:00:00 to 2026-06-30 00:00:00
Unique active users: 5000


,user_id,signup_date,country
0,1,2026-04-13,UK
1,2,2026-04-03,India
2,3,2026-01-15,India
3,4,2026-04-17,India
4,5,2026-03-13,Canada


,session_id,user_id,session_date
0,1,1,2026-05-27
1,2,1,2026-06-06
2,3,1,2026-06-16
3,4,1,2026-06-12
4,5,1,2026-06-08


,transaction_id,user_id,amount,transaction_date
0,1,295,139.87,2026-03-12
1,2,4172,177.09,2026-04-10
2,3,494,109.36,2026-06-05
3,4,3378,143.38,2026-06-26
4,5,2012,178.87,2026-04-11


## 3. SQLite Database Creation

The generated DataFrames are stored as three SQLite tables: `users`, `sessions`, and `transactions`.

In [4]:
# Create SQLite database connection
conn = sqlite3.connect("analytics.db")

# Write DataFrames into SQLite tables
users.to_sql("users", conn, if_exists="replace", index=False)
sessions.to_sql("sessions", conn, if_exists="replace", index=False)
transactions.to_sql("transactions", conn, if_exists="replace", index=False)

print("Database created successfully")

Database created successfully


In [5]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

pd.read_sql_query(query, conn)

,name
0,users
1,sessions
2,transactions


## 4. Ground-Truth Queries

Before introducing the LLM, a few analytics queries are executed manually. These provide verified reference values for later evaluation.

In [34]:
query = """
SELECT COUNT(DISTINCT user_id) AS dau
FROM sessions
WHERE DATE(session_date) = '2026-06-30';
"""

pd.read_sql_query(query, conn)

,dau
0,511


In [35]:
query = """
SELECT COUNT(DISTINCT s.user_id) AS us_dau
FROM sessions s
JOIN users u
    ON s.user_id = u.user_id
WHERE DATE(s.session_date) = '2026-06-30'
AND u.country = 'US';
"""

pd.read_sql_query(query, conn)

,us_dau
0,153


In [36]:
query = """
SELECT
    COUNT(DISTINCT u.user_id) AS cohort_size,
    COUNT(DISTINCT s.user_id) AS retained_users,
    ROUND(
        100.0 * COUNT(DISTINCT s.user_id)
        / COUNT(DISTINCT u.user_id),
        2
    ) AS d7_retention
FROM users u
LEFT JOIN sessions s
    ON u.user_id = s.user_id
    AND DATE(s.session_date) = DATE(u.signup_date, '+7 days')
WHERE DATE(u.signup_date) >= '2026-05-01'
AND DATE(u.signup_date) <= '2026-05-31';
"""

pd.read_sql_query(query, conn)

,cohort_size,retained_users,d7_retention
0,1067,207,19.4


### Additional Evaluation Ground Truths

The following manual queries establish reference values for comparison, revenue, and grouping questions used during evaluation.

In [37]:
# 1. Week-over-week average DAU
week_comparison_query = """
SELECT
  (SELECT AVG(daily_dau)
   FROM (
     SELECT COUNT(DISTINCT user_id) AS daily_dau
     FROM sessions
     WHERE DATE(session_date)
       BETWEEN DATE('2026-06-30', '-6 days')
       AND DATE('2026-06-30')
     GROUP BY DATE(session_date)
   )) AS avg_dau_this_week,

  (SELECT AVG(daily_dau)
   FROM (
     SELECT COUNT(DISTINCT user_id) AS daily_dau
     FROM sessions
     WHERE DATE(session_date)
       BETWEEN DATE('2026-06-30', '-13 days')
       AND DATE('2026-06-30', '-7 days')
     GROUP BY DATE(session_date)
   )) AS avg_dau_last_week;
"""

display(pd.read_sql_query(week_comparison_query, conn))


# 2. Total revenue in June 2026
revenue_query = """
SELECT ROUND(SUM(amount), 2) AS june_revenue
FROM transactions
WHERE DATE(transaction_date)
BETWEEN '2026-06-01' AND '2026-06-30';
"""

display(pd.read_sql_query(revenue_query, conn))


# 3. Country with most signups
country_query = """
SELECT country, COUNT(*) AS signups
FROM users
GROUP BY country
ORDER BY signups DESC
LIMIT 1;
"""

display(pd.read_sql_query(country_query, conn))

,avg_dau_this_week,avg_dau_last_week
0,486.571429,475.142857


,june_revenue
0,136380.17


,country,signups
0,US,1515


## 5. Gemini API Setup

The API key is loaded securely from Colab Secrets rather than being hard-coded in the notebook.

In [38]:
!pip install -q google-genai

In [39]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("API key loaded:", GEMINI_API_KEY is not None)

API key loaded: True


In [41]:
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client initialized successfully")

Gemini client initialized successfully


## 6. Schema Context and SQL Generation

The LLM receives an explicit database schema and metric definitions. It first decides whether a question is answerable or requires clarification. Clear questions are translated into read-only SQLite queries.

In [42]:
SCHEMA = """
Database: SQLite

Table: users
- user_id INTEGER
- signup_date DATETIME
- country TEXT

Table: sessions
- session_id INTEGER
- user_id INTEGER
- session_date DATETIME

Table: transactions
- transaction_id INTEGER
- user_id INTEGER
- amount REAL
- transaction_date DATETIME

Relationships:
sessions.user_id -> users.user_id
transactions.user_id -> users.user_id
"""

In [43]:
def clean_sql_output(text):
    text = text.strip()

    if text.startswith("```"):
        lines = text.splitlines()

        # Remove opening and closing markdown fences
        lines = lines[1:]

        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]

        text = "\n".join(lines).strip()

    return text

In [44]:
def generate_sql(question, model_name="gemini-2.5-flash"):
    prompt = f"""
You are a product analytics SQL assistant.

{SCHEMA}

First decide whether the user's analytics question is clear enough to answer.

If the question is ambiguous and different reasonable interpretations
would produce different answers, return:

CLARIFY
<one concise clarification question>

Otherwise return:

READY
<valid SQLite SELECT query>

Rules:
1. Use only tables and columns from the schema.
2. Never generate INSERT, UPDATE, DELETE, DROP, ALTER, or CREATE statements.
3. Use COUNT(DISTINCT user_id) for active-user metrics.
4. D7 retention means exact-day retention: a user is retained if they have
   a session exactly 7 days after signup.
5. The dataset ends on 2026-06-30. Interpret relative dates using this date.
6. Date columns contain timestamps. Always use DATE(column_name)
   when comparing dates.
7. Do not ask for clarification if the question has one clear analytical interpretation.

Question:
{question}
"""

    response = client.models.generate_content(
        model=model_name,
        contents=prompt
    )

    return response.text.strip()

## 7. SQL Safety and Execution

Generated SQL passes through a lightweight safety layer before execution. The prototype only permits `SELECT` queries and blocks common destructive SQL keywords.

In [45]:
def validate_sql(sql):
    cleaned_sql = sql.strip().upper()

    # Allow read-only SELECT queries and CTEs
    if not (
        cleaned_sql.startswith("SELECT")
        or cleaned_sql.startswith("WITH")
    ):
        return False, "Only read-only SELECT queries are allowed."

    blocked_keywords = [
        "INSERT", "UPDATE", "DELETE",
        "DROP", "ALTER", "CREATE",
        "REPLACE", "TRUNCATE"
    ]

    for keyword in blocked_keywords:
        if keyword in cleaned_sql:
            return False, f"Blocked SQL keyword detected: {keyword}"

    return True, "Query is safe."

In [46]:
def execute_sql(sql):
    is_safe, message = validate_sql(sql)

    if not is_safe:
        return None, f"Safety Error: {message}"

    try:
        result = pd.read_sql_query(sql, conn)
        return result, None

    except Exception as e:
        return None, str(e)

## 8. SQL Repair and Agent Pipeline

If a generated query fails during execution, the system sends the original question, failed SQL, and database error back to the model for one repair attempt. The retry is intentionally limited to control latency and API cost.

In [47]:
def repair_sql(question, broken_sql, error_message):
    prompt = f"""
You are fixing a SQLite query.

{SCHEMA}

Original question:
{question}

Broken SQL:
{broken_sql}

SQLite error:
{error_message}

Fix the query.

Rules:
1. Return ONLY the corrected SQL.
2. Use only tables and columns from the schema.
3. Return only a SELECT query.
4. Date columns contain timestamps. Use DATE(column_name) for date comparisons.
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return clean_sql_output(response.text)

In [48]:
def ask(question):
    print("Question:")
    print(question)

    response = generate_sql(question)

    if response.startswith("CLARIFY"):
        clarification = response.replace("CLARIFY", "", 1).strip()

        print("\nClarification needed:")
        print(clarification)

        return {
            "question": question,
            "status": "clarification_needed",
            "clarification": clarification
        }

    elif response.startswith("READY"):
        sql = response.replace("READY", "", 1).strip()

        print("\nGenerated SQL:")
        print(sql)

        result, error = execute_sql(sql)

        if error:
            print("\nInitial SQL failed. Retrying once...")

            repaired_sql = repair_sql(
                question,
                sql,
                error
            )

            print("\nRepaired SQL:")
            print(repaired_sql)

            result, retry_error = execute_sql(repaired_sql)

            if retry_error:
                print("\nQuery failed after retry:")
                print(retry_error)

                return {
                    "question": question,
                    "status": "failed",
                    "error": retry_error
                }

            sql = repaired_sql

        print("\nResult:")
        display(result)

        return {
            "question": question,
            "status": "success",
            "sql": sql,
            "result": result
        }

    else:
        print("Unexpected model response:")
        print(response)

        return {
            "question": question,
            "status": "error"
        }

## 9. Example Questions

The following examples demonstrate the required analytics behaviours: simple aggregates, filtered aggregates, cohort analysis, period comparison, and clarification for ambiguous questions.

In [49]:
ask("What was DAU on June 30, 2026?")

Question:
What was DAU on June 30, 2026?

Generated SQL:
SELECT
  COUNT(DISTINCT user_id)
FROM sessions
WHERE
  DATE(session_date) = '2026-06-30';

Result:


,COUNT(DISTINCT user_id)
0,511


{'question': 'What was DAU on June 30, 2026?',
 'status': 'success',
 'sql': "SELECT\n  COUNT(DISTINCT user_id)\nFROM sessions\nWHERE\n  DATE(session_date) = '2026-06-30';",
 'result':    COUNT(DISTINCT user_id)
 0                      511}

In [50]:
ask("What was DAU for US users on June 30, 2026?")

Question:
What was DAU for US users on June 30, 2026?

Generated SQL:
SELECT COUNT(DISTINCT s.user_id)
FROM sessions AS s
JOIN users AS u
  ON s.user_id = u.user_id
WHERE
  DATE(s.session_date) = '2026-06-30' AND u.country = 'US';

Result:


,COUNT(DISTINCT s.user_id)
0,153


{'question': 'What was DAU for US users on June 30, 2026?',
 'status': 'success',
 'sql': "SELECT COUNT(DISTINCT s.user_id)\nFROM sessions AS s\nJOIN users AS u\n  ON s.user_id = u.user_id\nWHERE\n  DATE(s.session_date) = '2026-06-30' AND u.country = 'US';",
 'result':    COUNT(DISTINCT s.user_id)
 0                        153}

In [51]:
 ask("What was the D7 retention rate for users who signed up in May 2026?")

Question:
What was the D7 retention rate for users who signed up in May 2026?

Generated SQL:
WITH UsersSignedUpInMay AS (
  SELECT
    user_id,
    signup_date
  FROM users
  WHERE
    DATE(signup_date) >= '2026-05-01' AND DATE(signup_date) < '2026-06-01'
),
RetainedUsers AS (
  SELECT DISTINCT
    usm.user_id
  FROM UsersSignedUpInMay AS usm
  JOIN sessions AS s
    ON usm.user_id = s.user_id
  WHERE
    DATE(s.session_date) = DATE(usm.signup_date, '+7 day')
)
SELECT
  CAST(COUNT(DISTINCT ru.user_id) AS REAL) * 100 / COUNT(DISTINCT usm.user_id)
FROM UsersSignedUpInMay AS usm
LEFT JOIN RetainedUsers AS ru
  ON usm.user_id = ru.user_id;

Result:


,CAST(COUNT(DISTINCT ru.user_id) AS REAL) * 100 / COUNT(DISTINCT usm.user_id)
0,19.400187


{'question': 'What was the D7 retention rate for users who signed up in May 2026?',
 'status': 'success',
 'sql': "WITH UsersSignedUpInMay AS (\n  SELECT\n    user_id,\n    signup_date\n  FROM users\n  WHERE\n    DATE(signup_date) >= '2026-05-01' AND DATE(signup_date) < '2026-06-01'\n),\nRetainedUsers AS (\n  SELECT DISTINCT\n    usm.user_id\n  FROM UsersSignedUpInMay AS usm\n  JOIN sessions AS s\n    ON usm.user_id = s.user_id\n  WHERE\n    DATE(s.session_date) = DATE(usm.signup_date, '+7 day')\n)\nSELECT\n  CAST(COUNT(DISTINCT ru.user_id) AS REAL) * 100 / COUNT(DISTINCT usm.user_id)\nFROM UsersSignedUpInMay AS usm\nLEFT JOIN RetainedUsers AS ru\n  ON usm.user_id = ru.user_id;",
 'result':    CAST(COUNT(DISTINCT ru.user_id) AS REAL) * 100 / COUNT(DISTINCT usm.user_id)
 0                                          19.400187                           }

In [52]:
ask("Compare average DAU this week versus last week.")

Question:
Compare average DAU this week versus last week.

Generated SQL:
WITH daily_active_users AS (
    SELECT
        DATE(session_date) AS activity_date,
        COUNT(DISTINCT user_id) AS dau
    FROM sessions
    WHERE
        DATE(session_date) <= '2026-06-30' -- Ensure data is within the dataset's end date
    GROUP BY
        1
),
weekly_dau_comparison AS (
    SELECT
        activity_date,
        dau,
        CASE
            WHEN activity_date BETWEEN DATE('2026-06-30', '-6 days') AND '2026-06-30' THEN 'This Week'
            WHEN activity_date BETWEEN DATE('2026-06-30', '-13 days') AND DATE('2026-06-30', '-7 days') THEN 'Last Week'
            ELSE NULL
        END AS week_period
    FROM daily_active_users
    WHERE
        -- Filter for the specific 14-day window to calculate DAU for both periods
        activity_date BETWEEN DATE('2026-06-30', '-13 days') AND '2026-06-30'
)
SELECT
    week_period,
    AVG(dau) AS average_dau
FROM weekly_dau_comparison
WHERE
    week_pe

,week_period,average_dau
0,Last Week,475.142857
1,This Week,486.571429


{'question': 'Compare average DAU this week versus last week.',
 'status': 'success',
 'sql': "WITH daily_active_users AS (\n    SELECT\n        DATE(session_date) AS activity_date,\n        COUNT(DISTINCT user_id) AS dau\n    FROM sessions\n    WHERE\n        DATE(session_date) <= '2026-06-30' -- Ensure data is within the dataset's end date\n    GROUP BY\n        1\n),\nweekly_dau_comparison AS (\n    SELECT\n        activity_date,\n        dau,\n        CASE\n            WHEN activity_date BETWEEN DATE('2026-06-30', '-6 days') AND '2026-06-30' THEN 'This Week'\n            WHEN activity_date BETWEEN DATE('2026-06-30', '-13 days') AND DATE('2026-06-30', '-7 days') THEN 'Last Week'\n            ELSE NULL\n        END AS week_period\n    FROM daily_active_users\n    WHERE\n        -- Filter for the specific 14-day window to calculate DAU for both periods\n        activity_date BETWEEN DATE('2026-06-30', '-13 days') AND '2026-06-30'\n)\nSELECT\n    week_period,\n    AVG(dau) AS average_d

In [53]:
ask("Who are our best users?")

Question:
Who are our best users?


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

## 10. Safety Check

A destructive SQL statement is rejected by the validation layer, while a valid read-only query executes normally.

In [54]:
execute_sql("DROP TABLE users;")

(None, 'Safety Error: Only read-only SELECT queries are allowed.')

In [55]:
execute_sql("""
SELECT COUNT(DISTINCT user_id) AS dau
FROM sessions
WHERE DATE(session_date) = '2026-06-30';
""")

(   dau
 0  511,
 None)

## 11. Evaluation

A small evaluation set tests query classification, SQL execution, numeric correctness where ground truth is available, and clarification behaviour for ambiguous questions.

In [56]:
eval_questions = [
    {
        "question": "What was DAU on June 30, 2026?",
        "type": "simple_aggregate",
        "expected_status": "success",
        "expected_value": 511
    },
    {
        "question": "What was DAU for US users on June 30, 2026?",
        "type": "filtered_aggregate",
        "expected_status": "success",
        "expected_value": 153
    },
    {
        "question": "What was the D7 retention rate for users who signed up in May 2026?",
        "type": "cohort",
        "expected_status": "success",
        "expected_value": 0.194
    },
    {
        "question": "Compare average DAU this week versus last week.",
        "type": "comparison",
        "expected_status": "success",
        "expected_value": None
    },
    {
        "question": "Who are our best users?",
        "type": "ambiguous",
        "expected_status": "clarification_needed",
        "expected_value": None
    },
    {
        "question": "How many users signed up in May 2026?",
        "type": "simple_aggregate",
        "expected_status": "success",
        "expected_value": 1067
    },
    {
        "question": "What was total revenue in June 2026?",
        "type": "aggregate",
        "expected_status": "success",
        "expected_value": 136380.17
    },
    {
        "question": "Which country had the most signups?",
        "type": "grouping",
        "expected_status": "success",
        "expected_value": None
    },
    {
        "question": "How engaged are our users?",
        "type": "ambiguous",
        "expected_status": "clarification_needed",
        "expected_value": None
    },
    {
        "question": "Are we doing well?",
        "type": "ambiguous",
        "expected_status": "clarification_needed",
        "expected_value": None
    }
]

In [57]:
def evaluate(eval_questions):
    rows = []

    for item in eval_questions:
        question = item["question"]
        expected_status = item["expected_status"]
        expected_value = item["expected_value"]

        # Call Gemini safely
        try:
            response = generate_sql(
                question,
                model_name="gemini-3.1-flash-lite"
            )

        except Exception as e:
            rows.append({
                "question": question,
                "type": item["type"],
                "expected_status": expected_status,
                "actual_status": "api_error",
                "status_correct": False,
                "sql_executed": False,
                "expected_value": expected_value,
                "actual_value": None,
                "numeric_correct": None
            })

            continue

        # Ambiguous question
        if response.startswith("CLARIFY"):
            actual_status = "clarification_needed"
            sql_executed = False
            actual_value = None

        # Answerable question
        elif response.startswith("READY"):
            sql = response.replace("READY", "", 1).strip()

            result, error = execute_sql(sql)

            # Retry once if generated SQL fails
            if error:
                try:
                    repaired_sql = repair_sql(
                        question,
                        sql,
                        error
                    )

                    result, error = execute_sql(repaired_sql)

                except Exception:
                    error = "API error during SQL repair"

            if error:
                actual_status = "failed"
                sql_executed = False
                actual_value = None

            else:
                actual_status = "success"
                sql_executed = True

                # Extract numeric value only for 1x1 results
                if result.shape == (1, 1):
                    actual_value = float(result.iloc[0, 0])
                else:
                    actual_value = None

        # Unexpected model response
        else:
            actual_status = "error"
            sql_executed = False
            actual_value = None

        # Check whether behaviour/status was correct
        status_correct = (
            actual_status == expected_status
        )

        # Check numeric correctness only where
        # a ground-truth value is available
        if (
            expected_value is not None
            and actual_value is not None
        ):
            numeric_correct = (
                abs(actual_value - expected_value) < 0.001
            )
        else:
            numeric_correct = None

        rows.append({
            "question": question,
            "type": item["type"],
            "expected_status": expected_status,
            "actual_status": actual_status,
            "status_correct": status_correct,
            "sql_executed": sql_executed,
            "expected_value": expected_value,
            "actual_value": actual_value,
            "numeric_correct": numeric_correct
        })

    return pd.DataFrame(rows)

In [31]:
eval_results = evaluate(eval_questions)
display(eval_results)

,question,type,expected_status,actual_status,status_correct,sql_executed,expected_value,actual_value,numeric_correct
0,"What was DAU on June 30, 2026?",simple_aggregate,success,success,True,True,511.000,511.000000,True
1,"What was DAU for US users on June 30, 2026?",filtered_aggregate,success,success,True,True,153.000,153.000000,True
2,What was the D7 retention rate for users who s...,cohort,success,success,True,True,0.194,0.186408,False
3,Compare average DAU this week versus last week.,comparison,success,success,True,True,NaN,NaN,None
4,Who are our best users?,ambiguous,clarification_needed,clarification_needed,True,False,NaN,NaN,None
5,How many users signed up in May 2026?,simple_aggregate,success,success,True,True,1067.000,1067.000000,True
6,What was total revenue in June 2026?,aggregate,success,success,True,True,136380.170,136380.170000,True
7,Which country had the most signups?,grouping,success,success,True,True,NaN,NaN,None
8,How engaged are our users?,ambiguous,clarification_needed,clarification_needed,True,False,NaN,NaN,None
9,Are we doing well?,ambiguous,clarification_needed,clarification_needed,True,False,NaN,NaN,None


In [32]:
print(
    "Status accuracy:",
    round(eval_results["status_correct"].mean() * 100, 1),
    "%"
)

answerable = eval_results[
    eval_results["expected_status"] == "success"
]

print(
    "SQL execution success:",
    round(answerable["sql_executed"].mean() * 100, 1),
    "%"
)

numeric_tests = eval_results[
    eval_results["numeric_correct"].notna()
]

print(
    "Numeric accuracy:",
    round(numeric_tests["numeric_correct"].mean() * 100, 1),
    "%"
)

Status accuracy: 100.0 %
SQL execution success: 100.0 %
Numeric accuracy: 80.0 %
